# Transient Leadership in Opinion-Contact Dynamics

Extensions of Albi, Calzola & Piu (2025) incorporating the **transient leadership**
framework from Albi, Almi, Morandotti & Solombrino (2022).

**Core idea:** In the original model, group labels (leader/mass) are fixed for all time.
We extend this so that labels evolve dynamically via a Markov-type jump process
whose transition rates depend on the agent's contacts. Leadership is *earned* through
popularity and *lost* when popularity declines.

Two experiments:
1. **Leader-follower system** — single leader group, compare fixed vs transient labels
2. **Polarized society** — two competing groups with fluid membership


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field, replace
from typing import Callable, Dict, List, Mapping, Optional, Sequence, Tuple
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.ndimage import gaussian_filter, gaussian_filter1d

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw): return it

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "figure.max_open_warning": 40})
print("Imports OK")

## Base Model (Albi et al. 2025, Table 1 + Algorithm 1)

In [ ]:
MASS = 0
LEADER = 1
LEADER_A = 1
LEADER_B = 2

@dataclass
class BaseParams:
    beta: float = 1.0
    mu: float = 0.0
    c_bar: float = 200.0
    theta: float = 2.0
    delta_phi: float = 0.1
    nu: float = 0.1
    gamma_c: float = 1.0
    lam: float = 1.0
    alpha_R: float = 0.1
    alpha_H: float = 0.1
    r: float = 0.7
    rho_star: float = 0.5
    c_min: float = 100.0
    alpha: float = 1.0
    delta: float = 0.8
    p: float = 3.0
    v_tilde: float = 0.5
    sigma: float = 0.1
    gamma_v: float = 10.0

@dataclass
class RuntimeConfig:
    T: float
    dt: float
    Ns: int
    pair_fraction: float = 1.0
    v_bins: int = 220
    c_bins: int = 220
    c_max: float = 300.0
    c_clip_max: Optional[float] = None
    rho_bins: int = 320
    c_floor: float = 1e-8
    eta_dist: str = "uniform"
    pair_mode: str = "matching"
    influence_mode: str = "partner"
    joint_smooth_sigma: float = 1.1
    marginal_smooth_sigma: float = 1.0
    seed: int = 1234

@dataclass
class ScenarioControl:
    key: str
    title: str
    contact_groups: Tuple[int, ...] = ()
    opinion_groups: Tuple[int, ...] = ()
    gamma_v_by_group: Dict[int, float] = field(default_factory=dict)
    target_by_group: Dict[int, float] = field(default_factory=dict)

def table1_params():
    return BaseParams()

def make_runtime(test_id, mode="fast", seed=1234):
    T = {1: 50.0, 2: 50.0, 3: 150.0}[test_id]
    cfgs = {
        "fast":     {1: (1e-2, 12_000), 2: (1e-2, 16_000), 3: (1e-2, 18_000)},
        "balanced": {1: (1e-2, 24_000), 2: (1e-2, 30_000), 3: (1e-2, 60_000)},
        "paper":    {1: (1e-3, 1_000_000), 2: (1e-3, 1_000_000), 3: (1e-3, 1_000_000)},
    }
    dt, Ns = cfgs[mode][test_id]
    return RuntimeConfig(T=T, dt=dt, Ns=Ns, seed=seed)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60.0, 60.0)))

def psi_eps(s, beta, mu, eps):
    s = np.maximum(s, 1e-12)
    if abs(mu) < 1e-14:
        return np.zeros_like(s)
    s_eps = np.power(s, eps)
    return (1.0/(beta*eps)) * (mu/(1.0-mu)) * (s_eps-1.0) / (((1.0+mu)/(1.0-mu))*s_eps + 1.0)

def phi_v(v, m_v, theta, delta_phi):
    return theta * ((v - m_v)**2 - delta_phi**2)

def Rc(c, alpha_R, c_min):
    return sigmoid(alpha_R * (c_min - c))

def Hc_from_rho(rho, alpha_H, rho_star):
    return sigmoid(alpha_H * (rho - rho_star))

def D(v):
    return 1.0 - v**2

class LocalMassEstimator:
    def __init__(self, r, n_bins=320, v_min=-1.0, v_max=1.0):
        self.r, self.n_bins = float(r), int(n_bins)
        self.edges = np.linspace(v_min, v_max, n_bins + 1)
        self.dv = self.edges[1] - self.edges[0]
        self.window = max(0, int(np.ceil(self.r / self.dv)))
    def __call__(self, v):
        idx = np.clip(np.searchsorted(self.edges, v, side="right") - 1, 0, self.n_bins - 1)
        counts = np.bincount(idx, minlength=self.n_bins).astype(np.float64)
        csum = np.concatenate(([0.0], np.cumsum(counts)))
        bins = np.arange(self.n_bins)
        left = np.maximum(bins - self.window, 0)
        right = np.minimum(bins + self.window + 1, self.n_bins)
        rho_bins = (csum[right] - csum[left]) / max(float(v.size), 1.0)
        return rho_bins[idx]

def compromise_terms(vi, vj, ci, cj, delta, p, mode="partner"):
    close = (np.abs(vi - vj) < delta).astype(np.float64)
    ci_p = np.power(np.maximum(ci, 1e-12), p)
    cj_p = np.power(np.maximum(cj, 1e-12), p)
    den = ci_p + cj_p + 1e-14
    if mode == "partner":
        K_ij = cj_p / den
        K_ji = ci_p / den
    else:
        K_ij = ci_p / den
        K_ji = cj_p / den
    return close * K_ij, close * K_ji

def make_edges(cfg):
    v_e = np.linspace(-1.0, 1.0, cfg.v_bins + 1)
    c_e = np.linspace(0.0, cfg.c_max, cfg.c_bins + 1)
    return v_e, c_e, 0.5*(v_e[:-1]+v_e[1:]), 0.5*(c_e[:-1]+c_e[1:])

def joint_density(v, c, v_edges, c_edges, smooth_sigma=0.0):
    H, _, _ = np.histogram2d(v, c, bins=[v_edges, c_edges], density=False)
    if smooth_sigma > 0:
        H = gaussian_filter(H, sigma=smooth_sigma, mode="nearest")
    return H / (max(float(v.size), 1.0) * (v_edges[1]-v_edges[0]) * (c_edges[1]-c_edges[0]))

def marginal_density(x, edges, smooth_sigma=0.0):
    h, _ = np.histogram(x, bins=edges, density=False)
    if smooth_sigma > 0:
        h = gaussian_filter1d(h.astype(np.float64), sigma=smooth_sigma, mode="nearest")
    return h / (max(float(x.size), 1.0) * (edges[1]-edges[0]))

def _time_map(times, dt):
    return {int(round(float(t)/dt)): float(t) for t in times}

print("Base model defined")

## Initial Conditions (Sections 4.1-4.2)

In [ ]:
def init_test1(Ns, seed=1234):
    rng = np.random.default_rng(seed)
    nl = int(round(0.25 * Ns)); nm = Ns - nl
    v, c, g = np.empty(Ns), np.empty(Ns), np.empty(Ns, dtype=np.int32)
    c[:nl] = rng.uniform(150, 200, nl); v[:nl] = rng.uniform(0.4, 0.6, nl); g[:nl] = LEADER
    c[nl:] = rng.uniform(10, 90, nm);   v[nl:] = rng.uniform(-0.9, -0.1, nm); g[nl:] = MASS
    p2 = rng.permutation(Ns); return v[p2], c[p2], g[p2]

def init_test2(Ns, seed=1234):
    rng = np.random.default_rng(seed)
    na = int(round(0.25*Ns)); nb = int(round(0.25*Ns)); nm = Ns - na - nb
    v, c, g = np.empty(Ns), np.empty(Ns), np.empty(Ns, dtype=np.int32)
    c[:na] = rng.uniform(200, 250, na); v[:na] = rng.uniform(-0.6, -0.4, na); g[:na] = LEADER_A
    s, e = na, na + nb
    c[s:e] = rng.uniform(200, 250, nb); v[s:e] = rng.uniform(0.4, 0.6, nb); g[s:e] = LEADER_B
    c[e:] = rng.uniform(50, 100, nm); v[e:] = rng.uniform(-0.8, 0.8, nm); g[e:] = MASS
    p2 = rng.permutation(Ns); return v[p2], c[p2], g[p2]

print("Initial conditions defined")

## Transient Leadership Simulation Engine

Key differences from the original Algorithm 1:
1. **Label update (Step 0):** Before each time step, group labels are updated
   via a Markov switching rule with contact-dependent transition rates.
2. **Control mask rebuild:** Control masks (`kappa*`, `u*`) are rebuilt from
   the *current* labels each step — not the initial labels.
3. **Carrying capacity:** A maximum leader fraction prevents runaway promotion.


In [ ]:
@dataclass
class TransientConfig:
    """Parameters for label switching dynamics."""
    # Contact-dependent transition rates
    rate_promote: float = 0.5       # max promotion rate (mass -> leader)
    rate_demote: float = 0.5        # max demotion rate (leader -> mass)
    alpha_T: float = 0.05           # sigmoid steepness for transition
    c_threshold: float = 80.0       # contacts midpoint for switching

    # Carrying capacity: maximum fraction of leaders allowed
    # When leader_frac >= max_leader_frac, no new promotions occur.
    max_leader_frac: float = 0.35

    # Two-group switching (Exp 2)
    v_attract_A: float = -0.5
    v_attract_B: float = 0.5
    opinion_width: float = 0.5      # half-width of opinion attractor
    rho_threshold: float = 0.2      # local mass threshold for group entry


def run_transient_sim(v0, c0, groups0, params, cfg, scenario,
                      snapshot_times, marginal_times,
                      tc, switch_rule="markov", progress=True):
    """Particle simulation with dynamic group labels.

    switch_rule:
      "markov"     - Stochastic Markov switching (Exp 1)
      "two_group"  - Opinion-based two-group switching (Exp 2)
    """
    rng = np.random.default_rng(cfg.seed)
    v = v0.copy().astype(np.float64)
    c = c0.copy().astype(np.float64)
    g = groups0.copy().astype(np.int32)
    N = v.size
    Nt = int(round(cfg.T / cfg.dt))
    eps = cfg.dt
    n_pairs = max(1, min(int(round(0.5 * cfg.pair_fraction * N)), N // 2))
    v_edges, c_edges, v_centers, c_centers = make_edges(cfg)
    snap_s = _time_map(snapshot_times, cfg.dt)
    marg_s = _time_map(marginal_times, cfg.dt)
    joint_snap, c_snap, v_snap = {}, {}, {}

    if 0 in snap_s:
        joint_snap[snap_s[0]] = joint_density(v, c, v_edges, c_edges, cfg.joint_smooth_sigma)
    if 0 in marg_s:
        c_snap[marg_s[0]] = marginal_density(c, c_edges, cfg.marginal_smooth_sigma)
        v_snap[marg_s[0]] = marginal_density(v, v_edges, cfg.marginal_smooth_sigma)

    mc, mv = np.empty(Nt+1), np.empty(Nt+1)
    leader_frac = np.empty(Nt+1)
    frac_A, frac_B = np.empty(Nt+1), np.empty(Nt+1)
    mc[0], mv[0] = np.mean(c), np.mean(v)
    leader_frac[0] = np.mean(g != MASS)
    frac_A[0] = np.mean(g == LEADER_A)
    frac_B[0] = np.mean(g == LEADER_B)

    rho_est = LocalMassEstimator(r=params.r, n_bins=cfg.rho_bins)
    sqrt_eps = np.sqrt(eps)
    it = tqdm(range(Nt), desc=f"[{switch_rule}]", leave=False) if progress else range(Nt)

    for n in it:
        m_v = float(np.mean(v))
        cur_leader_frac = np.mean(g != MASS)

        # ═══ STEP 0: UPDATE GROUP LABELS ═══
        if switch_rule == "markov":
            # Promotion: mass -> leader (contact-dependent, with carrying capacity)
            p_up = tc.rate_promote * sigmoid(tc.alpha_T * (c - tc.c_threshold)) * eps
            p_up = np.clip(p_up, 0, 1)
            # Block promotions if carrying capacity reached
            if cur_leader_frac >= tc.max_leader_frac:
                p_up[:] = 0.0

            # Demotion: leader -> mass (low contacts = lose status)
            p_dn = tc.rate_demote * sigmoid(tc.alpha_T * (tc.c_threshold - c)) * eps
            p_dn = np.clip(p_dn, 0, 1)

            coins = rng.uniform(size=N)
            g[(g == MASS) & (coins < p_up)] = LEADER
            coins = rng.uniform(size=N)
            g[(g == LEADER) & (coins < p_dn)] = MASS

        elif switch_rule == "two_group":
            rho_all = rho_est(v)
            near_A = np.abs(v - tc.v_attract_A) < tc.opinion_width
            near_B = np.abs(v - tc.v_attract_B) < tc.opinion_width
            high_c = c > tc.c_threshold
            high_rho = rho_all > tc.rho_threshold

            # Promotion (with carrying capacity per group)
            fA = np.mean(g == LEADER_A)
            fB = np.mean(g == LEADER_B)
            p_join = tc.rate_promote * eps

            if fA + fB < tc.max_leader_frac:
                coins = rng.uniform(size=N)
                g[(g == MASS) & near_A & high_c & high_rho & (coins < p_join)] = LEADER_A
                coins = rng.uniform(size=N)
                g[(g == MASS) & near_B & high_c & high_rho & (coins < p_join)] = LEADER_B

            # Demotion: lost opinion alignment OR lost contacts
            p_lose = tc.rate_demote * eps
            coins = rng.uniform(size=N)
            g[(g == LEADER_A) & (~near_A | ~high_c) & (coins < p_lose)] = MASS
            coins = rng.uniform(size=N)
            g[(g == LEADER_B) & (~near_B | ~high_c) & (coins < p_lose)] = MASS

        # ═══ STEP 1: REBUILD CONTROL MASKS FROM CURRENT LABELS ═══
        ce = np.isin(g, np.array(scenario.contact_groups, dtype=np.int32))
        oe = np.isin(g, np.array(scenario.opinion_groups, dtype=np.int32))
        gv_arr = np.full(N, params.gamma_v)
        tgt_arr = np.full(N, params.v_tilde)
        for k2, val in scenario.gamma_v_by_group.items():
            gv_arr[g == k2] = val
        for k2, val in scenario.target_by_group.items():
            tgt_arr[g == k2] = val

        # ═══ STEP 2: STANDARD Algorithm 1 UPDATE ═══
        perm = rng.permutation(N)
        paired = perm[:2*n_pairs].reshape(n_pairs, 2)
        i, j = paired[:, 0], paired[:, 1]
        vi, vj, ci, cj = v[i], v[j], c[i], c[j]

        phi_i = phi_v(vi, m_v, params.theta, params.delta_phi)
        phi_j = phi_v(vj, m_v, params.theta, params.delta_phi)

        # Contact control kappa*
        kappa_i, kappa_j = np.zeros(n_pairs), np.zeros(n_pairs)
        if np.any(ce):
            rho2 = rho_est(v)
            hc = Hc_from_rho(rho2, params.alpha_H, params.rho_star)
            kp = params.lam * params.beta / params.gamma_c
            kappa_i = kp * Rc(ci, params.alpha_R, params.c_min) * hc[i] * ce[i]
            kappa_j = kp * Rc(cj, params.alpha_R, params.c_min) * hc[j] * ce[j]

        Pij, Pji = compromise_terms(vi, vj, ci, cj, params.delta, params.p,
                                    mode=cfg.influence_mode)

        # Opinion control u*
        u_i, u_j = np.zeros(n_pairs), np.zeros(n_pairs)
        if np.any(oe):
            ti = vi + eps * params.alpha * Pij * (vj - vi) - tgt_arr[i]
            tj = vj + eps * params.alpha * Pji * (vi - vj) - tgt_arr[j]
            u_i = -ti / (gv_arr[i] + eps * params.alpha) * oe[i]
            u_j = -tj / (gv_arr[j] + eps * params.alpha) * oe[j]

        # Noise
        amp = np.sqrt(3.0) * sqrt_eps * params.nu
        eta_i = amp * rng.uniform(-1, 1, n_pairs)
        eta_j = amp * rng.uniform(-1, 1, n_pairs)
        xi_i, xi_j = rng.normal(size=n_pairs), rng.normal(size=n_pairs)

        # Update contacts
        psi_i = psi_eps(ci / params.c_bar, params.beta, params.mu, eps)
        psi_j = psi_eps(cj / params.c_bar, params.beta, params.mu, eps)
        ci_new = ci - eps*params.beta*(psi_i + phi_i - kappa_i)*ci + eta_i*ci
        cj_new = cj - eps*params.beta*(psi_j + phi_j - kappa_j)*cj + eta_j*cj
        c[i] = np.maximum(ci_new, cfg.c_floor)
        c[j] = np.maximum(cj_new, cfg.c_floor)

        # Update opinions
        v[i] = np.clip(vi + eps*params.alpha*(Pij*(vj-vi)+u_i) + sqrt_eps*params.sigma*D(vi)*xi_i, -1, 1)
        v[j] = np.clip(vj + eps*params.alpha*(Pji*(vi-vj)+u_j) + sqrt_eps*params.sigma*D(vj)*xi_j, -1, 1)

        mc[n+1], mv[n+1] = np.mean(c), np.mean(v)
        leader_frac[n+1] = np.mean(g != MASS)
        frac_A[n+1] = np.mean(g == LEADER_A)
        frac_B[n+1] = np.mean(g == LEADER_B)

        step = n + 1
        if step in snap_s:
            joint_snap[snap_s[step]] = joint_density(v, c, v_edges, c_edges, cfg.joint_smooth_sigma)
        if step in marg_s:
            c_snap[marg_s[step]] = marginal_density(c, c_edges, cfg.marginal_smooth_sigma)
            v_snap[marg_s[step]] = marginal_density(v, v_edges, cfg.marginal_smooth_sigma)

    return {"times": np.arange(Nt+1)*cfg.dt, "mc": mc, "mv": mv,
            "leader_frac": leader_frac, "frac_A": frac_A, "frac_B": frac_B,
            "joint": joint_snap, "contact_marginals": c_snap, "opinion_marginals": v_snap,
            "v_edges": v_edges, "c_edges": c_edges,
            "v_centers": v_centers, "c_centers": c_centers,
            "final_v": v, "final_c": c, "groups": g}

print("Transient simulation engine defined (with carrying capacity fix)")

## Experiment 1: Transient Leadership in Leader-Follower System

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  Experiment 1: Transient leadership in a leader-follower system
# ════════════════════════════════════════════════════════════════════
#
#  Motivation (social media analogy):
#    On social media, influencer status is NOT permanent.  An account
#    gains influence when it accumulates followers (contacts), but
#    loses it when followers leave.  A platform's moderation policy
#    (= control) targets whoever the current influencers are.
#
#  Setup: Test 1 initial condition (25% leaders, 75% mass).
#    Joint control (kappa* + u*) is active on whoever is currently
#    labeled LEADER.  Three scenarios compared:
#
#    (a) Fixed — labels never change (= original Test 1, scenario d)
#    (b) Transient — Markov switching with carrying capacity 35%
#    (c) Transient (fast) — higher switching rate
#
#  Key question: Does allowing leadership to be earned/lost through
#  popularity improve or hurt the controller's ability to steer
#  opinions toward the target?
# ════════════════════════════════════════════════════════════════════

def run_experiment_1(mode="fast", seed=1234, progress=True):
    params = table1_params()
    cfg = make_runtime(1, mode=mode, seed=seed)
    v0, c0, g0 = init_test1(cfg.Ns, cfg.seed)
    v_edges, c_edges, v_centers, c_centers = make_edges(cfg)
    ij = joint_density(v0, c0, v_edges, c_edges, cfg.joint_smooth_sigma)

    sc = ScenarioControl(
        key="d", title="Joint control",
        contact_groups=(LEADER,), opinion_groups=(LEADER,),
        gamma_v_by_group={LEADER: params.gamma_v},
        target_by_group={LEADER: params.v_tilde})

    snap_t = [1.0, 5.0, 15.0, 50.0]
    marg_t = [0.0, 1.0, 5.0, 15.0, 50.0]
    results = {}

    # (a) Fixed: unreachable thresholds = no switching
    tc_fix = TransientConfig(rate_promote=0, rate_demote=0, max_leader_frac=1.0)
    cfg_a = replace(cfg, seed=cfg.seed + 100)
    results["fixed"] = run_transient_sim(
        v0, c0, g0, params, cfg_a, sc, snap_t, marg_t,
        tc_fix, "markov", progress)

    # (b) Transient (moderate switching)
    tc_mod = TransientConfig(
        rate_promote=0.5, rate_demote=0.5,
        alpha_T=0.05, c_threshold=80.0,
        max_leader_frac=0.35)
    cfg_b = replace(cfg, seed=cfg.seed + 200)
    results["transient"] = run_transient_sim(
        v0, c0, g0, params, cfg_b, sc, snap_t, marg_t,
        tc_mod, "markov", progress)

    # (c) Transient (fast switching)
    tc_fast = TransientConfig(
        rate_promote=2.0, rate_demote=2.0,
        alpha_T=0.05, c_threshold=80.0,
        max_leader_frac=0.35)
    cfg_c = replace(cfg, seed=cfg.seed + 300)
    results["fast"] = run_transient_sim(
        v0, c0, g0, params, cfg_c, sc, snap_t, marg_t,
        tc_fast, "markov", progress)

    return {"config": cfg, "params": params, "results": results,
            "v0": v0, "c0": c0, "groups0": g0, "initial_joint": ij,
            "v_edges": v_edges, "c_edges": c_edges,
            "v_centers": v_centers, "c_centers": c_centers}


def plot_experiment_1(exp):
    colors = {"fixed": "tab:blue", "transient": "tab:orange", "fast": "tab:red"}
    labels = {"fixed": "Fixed labels", "transient": "Transient (mod.)", "fast": "Transient (fast)"}

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))

    for key, res in exp["results"].items():
        t = res["times"]
        axes[0,0].plot(t, res["mc"], color=colors[key], lw=2, label=labels[key])
        axes[0,1].plot(t, res["mv"], color=colors[key], lw=2, label=labels[key])
        axes[0,2].plot(t, res["leader_frac"], color=colors[key], lw=2, label=labels[key])

    axes[0,0].set_xlabel("$t$"); axes[0,0].set_ylabel("$m_c(t)$")
    axes[0,0].set_title("Mean contacts"); axes[0,0].legend(); axes[0,0].grid(alpha=0.25)
    axes[0,1].set_xlabel("$t$"); axes[0,1].set_ylabel("$m_v(t)$")
    axes[0,1].set_title("Mean opinion"); axes[0,1].axhline(0.5, color="gray", ls="--", lw=1)
    axes[0,1].legend(); axes[0,1].grid(alpha=0.25)
    axes[0,2].set_xlabel("$t$"); axes[0,2].set_ylabel("Fraction")
    axes[0,2].set_title("Leader fraction"); axes[0,2].legend(); axes[0,2].grid(alpha=0.25)

    v_e, c_e = exp["v_edges"], exp["c_edges"]
    last_t = max(exp["results"]["fixed"]["joint"].keys())
    for idx, key in enumerate(["fixed", "transient", "fast"]):
        res = exp["results"][key]
        axes[1,idx].pcolormesh(v_e, c_e, res["joint"][last_t].T, cmap="turbo", shading="auto")
        axes[1,idx].set_xlabel("$v$"); axes[1,idx].set_ylabel("$c$")
        axes[1,idx].set_title(f"{labels[key]} | t={last_t:g}")

    fig.suptitle("Experiment 1: Transient leadership in leader-follower system",
                 fontsize=14, y=1.01)
    fig.tight_layout()
    return fig

print("Experiment 1 defined")

## Experiment 2: Transient Group Loyalty in Polarized Society

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  Experiment 2: Transient group loyalty in a polarized society
# ════════════════════════════════════════════════════════════════════
#
#  Motivation (political polarization):
#    In a polarized society, individuals may switch ideological
#    camps.  People near opinion pole A who become popular enough
#    join Group A; those near pole B join Group B.  But if they
#    lose followers or change their mind, they revert to the
#    undecided mass.  A policy maker applies BOTH contact control
#    (maintaining visibility) and opinion control (steering toward
#    target) on whichever group an agent currently belongs to.
#
#  Setup: Test 2 initial condition (two leader groups + mass).
#    BOTH controls active (kappa* + u*) — this is the key fix
#    compared to the old Experiment C which had no contact control.
#
#    (a) Fixed — labels never change, symmetric joint control
#    (b) Transient — agents can switch mass <-> A/B dynamically
#
#  Key question: Does fluid group membership amplify or dampen
#  polarization when both controls are active?
# ════════════════════════════════════════════════════════════════════

def run_experiment_2(mode="fast", seed=3333, progress=True):
    params = table1_params()
    cfg = make_runtime(2, mode=mode, seed=seed)
    v0, c0, g0 = init_test2(cfg.Ns, cfg.seed)
    v_edges, c_edges, v_centers, c_centers = make_edges(cfg)
    ij = joint_density(v0, c0, v_edges, c_edges, cfg.joint_smooth_sigma)

    # KEY FIX: contact_groups includes BOTH leader groups
    sc = ScenarioControl(
        key="sym_joint", title="Symmetric joint control",
        contact_groups=(LEADER_A, LEADER_B),   # <-- was () in old code!
        opinion_groups=(LEADER_A, LEADER_B),
        gamma_v_by_group={LEADER_A: 1.0, LEADER_B: 1.0},
        target_by_group={LEADER_A: -0.5, LEADER_B: 0.5})

    snap_t = [1.0, 5.0, 15.0, 50.0]
    marg_t = [0.0, 1.0, 5.0, 15.0, 50.0]
    results = {}

    # (a) Fixed labels
    tc_fix = TransientConfig(rate_promote=0, rate_demote=0, max_leader_frac=1.0)
    cfg_a = replace(cfg, seed=cfg.seed + 100)
    results["fixed"] = run_transient_sim(
        v0, c0, g0, params, cfg_a, sc, snap_t, marg_t,
        tc_fix, "two_group", progress)

    # (b) Transient group loyalty
    tc_trans = TransientConfig(
        rate_promote=0.5, rate_demote=0.3,
        alpha_T=0.05, c_threshold=60.0,
        v_attract_A=-0.5, v_attract_B=0.5,
        opinion_width=0.5,      # wider basin than before (was 0.4)
        rho_threshold=0.2,      # lower threshold (was 0.3)
        max_leader_frac=0.55)   # allow up to 55% total leaders
    cfg_b = replace(cfg, seed=cfg.seed + 200)
    results["transient"] = run_transient_sim(
        v0, c0, g0, params, cfg_b, sc, snap_t, marg_t,
        tc_trans, "two_group", progress)

    return {"config": cfg, "params": params, "results": results,
            "v0": v0, "c0": c0, "groups0": g0, "initial_joint": ij,
            "v_edges": v_edges, "c_edges": c_edges,
            "v_centers": v_centers, "c_centers": c_centers}


def plot_experiment_2(exp):
    rf = exp["results"]["fixed"]
    rt = exp["results"]["transient"]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))

    # Row 0: time series
    for label, res, st in [("Fixed", rf, "b-"), ("Transient", rt, "r--")]:
        t = res["times"]
        axes[0,0].plot(t, res["mc"], st, lw=2, label=label)
        axes[0,1].plot(t, res["mv"], st, lw=2, label=label)
        axes[0,2].plot(t, res["frac_A"], st, lw=2, label=f"{label} A")
        axes[0,2].plot(t, res["frac_B"], st[0]+":", lw=2, label=f"{label} B", alpha=0.7)

    axes[0,0].set_xlabel("$t$"); axes[0,0].set_ylabel("$m_c(t)$")
    axes[0,0].set_title("Mean contacts"); axes[0,0].legend(); axes[0,0].grid(alpha=0.25)
    axes[0,1].set_xlabel("$t$"); axes[0,1].set_ylabel("$m_v(t)$")
    axes[0,1].set_title("Mean opinion"); axes[0,1].legend(); axes[0,1].grid(alpha=0.25)
    axes[0,2].set_xlabel("$t$"); axes[0,2].set_ylabel("Fraction")
    axes[0,2].set_title("Group A / B fractions")
    axes[0,2].legend(fontsize=8); axes[0,2].grid(alpha=0.25)

    # Row 1: joint densities + opinion marginal
    v_e, c_e = exp["v_edges"], exp["c_edges"]
    last_t = max(rf["joint"].keys())
    axes[1,0].pcolormesh(v_e, c_e, rf["joint"][last_t].T, cmap="turbo", shading="auto")
    axes[1,0].set_xlabel("$v$"); axes[1,0].set_ylabel("$c$")
    axes[1,0].set_title(f"Fixed | t={last_t:g}")
    axes[1,1].pcolormesh(v_e, c_e, rt["joint"][last_t].T, cmap="turbo", shading="auto")
    axes[1,1].set_xlabel("$v$"); axes[1,1].set_ylabel("$c$")
    axes[1,1].set_title(f"Transient | t={last_t:g}")

    vc = exp["v_centers"]
    axes[1,2].plot(vc, rf["opinion_marginals"][last_t], "b-", lw=2, label="Fixed")
    axes[1,2].plot(vc, rt["opinion_marginals"][last_t], "r--", lw=2, label="Transient")
    axes[1,2].set_xlabel("$v$"); axes[1,2].set_ylabel("$g(v)$")
    axes[1,2].set_title(f"Opinion marginal | t={last_t:g}")
    axes[1,2].legend()

    fig.suptitle("Experiment 2: Transient group loyalty in polarized society",
                 fontsize=14, y=1.01)
    fig.tight_layout()
    return fig

print("Experiment 2 defined")

---
## Run Experiment 1
Test 1 setup: 25% leaders (high contacts, opinion ~0.5), 75% mass (low contacts, opinion ~-0.5).
Joint control targets opinion 0.5. Compare fixed, moderate-transient, fast-transient.
Change `mode="balanced"` for smoother results (~5 min).

In [ ]:
exp1 = run_experiment_1(mode="fast")
plot_experiment_1(exp1)
plt.show()

---
## Run Experiment 2
Test 2 setup: two leader groups (A near -0.5, B near +0.5), mass spread across opinions.
**Both** contact and opinion control are active (this was the bug fix from the previous version).
Compare fixed labels vs transient group membership.

In [ ]:
exp2 = run_experiment_2(mode="fast")
plot_experiment_2(exp2)
plt.show()

---
## Summary

Print key metrics for both experiments.

In [ ]:
print("=" * 65)
print("  Experiment 1: Leader-Follower Transient Leadership")
print("=" * 65)
for key in ["fixed", "transient", "fast"]:
    r = exp1["results"][key]
    print(f"  {key:12s}:  mc={r['mc'][-1]:6.1f}  mv={r['mv'][-1]:+.4f}  "
          f"leader_frac={r['leader_frac'][-1]:.3f}")

print()
print("=" * 65)
print("  Experiment 2: Polarized Society Transient Groups")
print("=" * 65)
for key in ["fixed", "transient"]:
    r = exp2["results"][key]
    print(f"  {key:12s}:  mc={r['mc'][-1]:6.1f}  mv={r['mv'][-1]:+.4f}  "
          f"frac_A={r['frac_A'][-1]:.3f}  frac_B={r['frac_B'][-1]:.3f}")